# Preparción de datos para PCA y aprendiaje automático

Este notebook construye una rama de modelado a partir de la base física procesada de las Gemínidas. 

In [5]:
# Librerías / Herramientas

from pathlib import Path

import numpy as np
import pandas as pd

## Abrir y auditar la base inicial

In [6]:
# Localizar la raíz del proyecto

directorio_actual = Path.cwd().resolve()
carpetas_requeridas = ["data", "notebooks", "src", "reports", "references"]
archivos_requeridos = ["README.md", "pyproject.toml", "uv.lock"]

raiz_proyecto = None

for posible_raiz in [directorio_actual, *directorio_actual.parents]:
    tiene_carpetas = all((posible_raiz / nombre).is_dir() for nombre in carpetas_requeridas)
    tiene_archivos = all((posible_raiz / nombre).is_file() for nombre in archivos_requeridos)

    if tiene_carpetas and tiene_archivos:
        raiz_proyecto = posible_raiz
        break

if raiz_proyecto is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto Investigacion.")

print("Directorio actual:", directorio_actual)
print("Raíz del proyecto:", raiz_proyecto)

Directorio actual: /Users/alexn/Desktop/GitHub/Investigacion_geminidas/notebooks
Raíz del proyecto: /Users/alexn/Desktop/GitHub/Investigacion_geminidas


In [7]:
# Definir y comprobar las rutas de entrada y salida

ruta_base_fisica = raiz_proyecto / "data" / "processed" / "geminidas_v1" / "geminidas_estandar_analisis_v1.parquet"
ruta_diccionario_fisico = raiz_proyecto / "data" / "processed" / "geminidas_v1" / "metadata" / "diccionario_variables_geminidas_v1.csv"
carpeta_modelado = raiz_proyecto / "data" / "processed" / "geminidas_modelado_v2"

if not ruta_base_fisica.is_file():
    raise FileNotFoundError(f"No se encontró la base física: {ruta_base_fisica}")

if not ruta_diccionario_fisico.is_file():
    raise FileNotFoundError(f"No se encontró el diccionario: {ruta_diccionario_fisico}")

print("Base física:", ruta_base_fisica)
print("Diccionario:", ruta_diccionario_fisico)
print("Salida futura:", carpeta_modelado)

Base física: /Users/alexn/Desktop/GitHub/Investigacion_geminidas/data/processed/geminidas_v1/geminidas_estandar_analisis_v1.parquet
Diccionario: /Users/alexn/Desktop/GitHub/Investigacion_geminidas/data/processed/geminidas_v1/metadata/diccionario_variables_geminidas_v1.csv
Salida futura: /Users/alexn/Desktop/GitHub/Investigacion_geminidas/data/processed/geminidas_modelado_v2


In [8]:
# Cargar la base física y su diccionario de variables

df_base_modelado = pd.read_parquet(ruta_base_fisica)
df_diccionario_fisico = pd.read_csv(ruta_diccionario_fisico)

print("Dimensiones de la base:", df_base_modelado.shape)
print("Variables documentadas:", len(df_diccionario_fisico))
print("Años presentes:", sorted(df_base_modelado["source_year"].unique().tolist()))

Dimensiones de la base: (19079, 51)
Variables documentadas: 51
Años presentes: [2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [9]:
# Comprobar que la base contiene las variables y observaciones necesarias

columnas_necesarias = [
    "id_fila_origen",
    "Beginning (UTC Time)",
    "source_year",
    "delta_sol_lon_deg",
    "lambda_sce_deg",
    "BETgeo (deg)",
    "Vgeo (km/s)",
    "magnitud_analisis",
    "fotometria_utilizable",
    "pertenece_muestra_estricta",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Participating (stations)",
    "sigma_radiante_aprox",
    "Vgeo_sigma"
]

columnas_faltantes = [columna for columna in columnas_necesarias if columna not in df_base_modelado.columns]

if columnas_faltantes:
    raise KeyError(f"Faltan variables necesarias: {columnas_faltantes}")

resumen_anual = df_base_modelado.groupby("source_year", as_index=False).agg(
    trayectorias=("id_fila_origen", "size"),
    fotometrias_utilizables=("fotometria_utilizable", "sum"),
    trayectorias_estrictas=("pertenece_muestra_estricta", "sum")
)

resumen_integridad = pd.DataFrame([
    {"comprobacion": "Filas totales", "resultado": len(df_base_modelado)},
    {"comprobacion": "Identificadores únicos", "resultado": df_base_modelado["id_fila_origen"].is_unique},
    {"comprobacion": "Columnas necesarias ausentes", "resultado": len(columnas_faltantes)},
    {"comprobacion": "Fotometrías utilizables", "resultado": int(df_base_modelado["fotometria_utilizable"].sum())},
    {"comprobacion": "Trayectorias estrictas", "resultado": int(df_base_modelado["pertenece_muestra_estricta"].sum())}
])

display(resumen_integridad)
display(resumen_anual)

,comprobacion,resultado
0,Filas totales,19079
1,Identificadores únicos,True
2,Columnas necesarias ausentes,0
3,Fotometrías utilizables,19054
4,Trayectorias estrictas,7040


,source_year,trayectorias,fotometrias_utilizables,trayectorias_estrictas
0,2019,430,430,99
1,2020,1002,1002,174
2,2021,1904,1903,293
3,2022,3712,3688,1755
4,2023,3397,3397,1267
5,2024,2459,2459,578
6,2025,6175,6175,2874


## Separar entrenamiento y prueba

In [10]:
# Separar formalmente 2019–2024 como entrenamiento y 2025 como prueba

anios_esperados = set(range(2019, 2026))
anios_observados = set(df_base_modelado["source_year"].unique())

if anios_observados != anios_esperados:
    raise ValueError(f"Los años observados no coinciden con los esperados: {sorted(anios_observados)}")

df_modelado = df_base_modelado.copy()
df_modelado["particion_modelado"] = np.where(df_modelado["source_year"] <= 2024, "entrenamiento", "prueba")

df_entrenamiento = df_modelado.loc[df_modelado["particion_modelado"] == "entrenamiento"].copy()
df_prueba = df_modelado.loc[df_modelado["particion_modelado"] == "prueba"].copy()

ids_entrenamiento = set(df_entrenamiento["id_fila_origen"])
ids_prueba = set(df_prueba["id_fila_origen"])
ids_totales = set(df_modelado["id_fila_origen"])

resumen_particiones = pd.DataFrame([
    {
        "particion": "Entrenamiento",
        "anios": sorted(df_entrenamiento["source_year"].unique().tolist()),
        "trayectorias": len(df_entrenamiento),
        "fotometrias_utilizables": int(df_entrenamiento["fotometria_utilizable"].sum()),
        "trayectorias_estrictas": int(df_entrenamiento["pertenece_muestra_estricta"].sum())
    },
    {
        "particion": "Prueba",
        "anios": sorted(df_prueba["source_year"].unique().tolist()),
        "trayectorias": len(df_prueba),
        "fotometrias_utilizables": int(df_prueba["fotometria_utilizable"].sum()),
        "trayectorias_estrictas": int(df_prueba["pertenece_muestra_estricta"].sum())
    }
])

auditoria_particiones = pd.DataFrame([
    {"comprobacion": "Sin identificadores compartidos", "resultado": len(ids_entrenamiento & ids_prueba) == 0},
    {"comprobacion": "Todas las filas fueron asignadas", "resultado": ids_entrenamiento | ids_prueba == ids_totales},
    {"comprobacion": "Entrenamiento termina en 2024", "resultado": df_entrenamiento["source_year"].max() == 2024},
    {"comprobacion": "Prueba contiene solo 2025", "resultado": set(df_prueba["source_year"].unique()) == {2025}}
])

display(resumen_particiones)
display(auditoria_particiones)

,particion,anios,trayectorias,fotometrias_utilizables,trayectorias_estrictas
0,Entrenamiento,"[2019, 2020, 2021, 2022, 2023, 2024]",12904,12879,4166
1,Prueba,[2025],6175,6175,2874


,comprobacion,resultado
0,Sin identificadores compartidos,True
1,Todas las filas fueron asignadas,True
2,Entrenamiento termina en 2024,True
3,Prueba contiene solo 2025,True


## Ajustar el centro físico con entrenamiento

In [11]:
# Auditar las variables que se utilizarán para ajustar el centro con entrenamiento

columnas_ajuste_centro = ["delta_sol_lon_deg", "lambda_sce_deg", "BETgeo (deg)", "Vgeo (km/s)"]

filas_auditoria_centro = []

for columna in columnas_ajuste_centro:
    valores = pd.to_numeric(df_entrenamiento[columna], errors="coerce")

    filas_auditoria_centro.append({
        "variable": columna,
        "trayectorias": len(valores),
        "valores_faltantes": int(valores.isna().sum()),
        "valores_no_finitos": int((~np.isfinite(valores.dropna())).sum()),
        "minimo": float(valores.min()),
        "mediana": float(valores.median()),
        "maximo": float(valores.max())
    })

df_auditoria_centro = pd.DataFrame(filas_auditoria_centro)

if df_auditoria_centro["valores_faltantes"].sum() != 0:
    raise ValueError("Existen valores faltantes en las variables del ajuste temporal.")

if df_auditoria_centro["valores_no_finitos"].sum() != 0:
    raise ValueError("Existen valores no finitos en las variables del ajuste temporal.")

display(df_auditoria_centro)

,variable,trayectorias,valores_faltantes,valores_no_finitos,minimo,mediana,maximo
0,delta_sol_lon_deg,12904,0,0,-4.995067,-0.314424,1.999240
1,lambda_sce_deg,12904,0,0,206.037673,208.054705,209.991913
2,BETgeo (deg),12904,0,0,8.582500,10.528195,12.420020
3,Vgeo (km/s),12904,0,0,30.280370,33.881860,37.489670


In [12]:
# Ajustar los modelos temporales usando exclusivamente las trayectorias de 2019–2024

def ajustar_polinomio(x, y, grado):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    coeficientes = np.polynomial.polynomial.polyfit(x, y, grado)
    valores_ajustados = np.polynomial.polynomial.polyval(x, coeficientes)
    residuos = y - valores_ajustados

    suma_cuadrados_residuos = np.sum(residuos ** 2)
    suma_cuadrados_total = np.sum((y - np.mean(y)) ** 2)
    r_cuadrada = 1 - suma_cuadrados_residuos / suma_cuadrados_total
    rmse = np.sqrt(np.mean(residuos ** 2))

    return coeficientes, r_cuadrada, rmse

configuracion_centro = {
    "lambda_sce": {"columna": "lambda_sce_deg", "grado": 1},
    "beta_geo": {"columna": "BETgeo (deg)", "grado": 2},
    "vgeo": {"columna": "Vgeo (km/s)", "grado": 1}
}

x_entrenamiento = df_entrenamiento["delta_sol_lon_deg"].to_numpy(dtype=float)
ajustes_centro_train = {}
filas_parametros_centro = []

for variable, configuracion in configuracion_centro.items():
    y_entrenamiento = df_entrenamiento[configuracion["columna"]].to_numpy(dtype=float)
    coeficientes, r_cuadrada, rmse = ajustar_polinomio(x_entrenamiento, y_entrenamiento, configuracion["grado"])

    ajustes_centro_train[variable] = {
        "columna": configuracion["columna"],
        "grado": configuracion["grado"],
        "coeficientes": coeficientes
    }

    filas_parametros_centro.append({
        "variable": variable,
        "modelo": "Cuadrático" if configuracion["grado"] == 2 else "Lineal",
        "trayectorias_entrenamiento": len(y_entrenamiento),
        "intercepto": coeficientes[0],
        "coeficiente_lineal": coeficientes[1],
        "coeficiente_cuadratico": coeficientes[2] if configuracion["grado"] == 2 else 0.0,
        "r_cuadrada": r_cuadrada,
        "rmse": rmse
    })

df_parametros_centro_train = pd.DataFrame(filas_parametros_centro)

display(df_parametros_centro_train)

,variable,modelo,trayectorias_entrenamiento,intercepto,coeficiente_lineal,coeficiente_cuadratico,r_cuadrada,rmse
0,lambda_sce,Lineal,12904,207.993531,-0.107974,0.000000,0.084678,0.395654
1,beta_geo,Cuadrático,12904,10.494435,-0.108343,-0.018288,0.035870,0.379410
2,vgeo,Lineal,12904,33.938456,0.110633,0.000000,0.040409,0.600880


In [13]:
# Aplicar los modelos aprendidos con 2019–2024 a todas las trayectorias

df_modelado_trainseguro = df_modelado.copy()
x_todos = df_modelado_trainseguro["delta_sol_lon_deg"].to_numpy(dtype=float)

df_modelado_trainseguro["lambda_sce_esperada_train_deg"] = np.polynomial.polynomial.polyval(x_todos, ajustes_centro_train["lambda_sce"]["coeficientes"])
df_modelado_trainseguro["beta_esperada_train_deg"] = np.polynomial.polynomial.polyval(x_todos, ajustes_centro_train["beta_geo"]["coeficientes"])
df_modelado_trainseguro["vgeo_esperada_train_km_s"] = np.polynomial.polynomial.polyval(x_todos, ajustes_centro_train["vgeo"]["coeficientes"])

df_modelado_trainseguro["residuo_lambda_train_deg"] = (df_modelado_trainseguro["lambda_sce_deg"] - df_modelado_trainseguro["lambda_sce_esperada_train_deg"] + 180) % 360 - 180
df_modelado_trainseguro["residuo_beta_train_deg"] = df_modelado_trainseguro["BETgeo (deg)"] - df_modelado_trainseguro["beta_esperada_train_deg"]
df_modelado_trainseguro["residuo_vgeo_train_km_s"] = df_modelado_trainseguro["Vgeo (km/s)"] - df_modelado_trainseguro["vgeo_esperada_train_km_s"]

df_entrenamiento_trainseguro = df_modelado_trainseguro.loc[df_modelado_trainseguro["particion_modelado"] == "entrenamiento"].copy()
df_prueba_trainseguro = df_modelado_trainseguro.loc[df_modelado_trainseguro["particion_modelado"] == "prueba"].copy()

In [14]:
# Comparar los residuos del centro físico entre entrenamiento y prueba

configuracion_residuos = {
    "Longitud SCE": "residuo_lambda_train_deg",
    "Latitud eclíptica": "residuo_beta_train_deg",
    "Velocidad geocéntrica": "residuo_vgeo_train_km_s"
}

filas_resumen_residuos = []

for particion, df_particion in df_modelado_trainseguro.groupby("particion_modelado", observed=True):
    for variable, columna in configuracion_residuos.items():
        residuos = df_particion[columna].to_numpy(dtype=float)

        filas_resumen_residuos.append({
            "particion": particion,
            "variable": variable,
            "trayectorias": len(residuos),
            "residuo_medio": np.mean(residuos),
            "residuo_mediano": np.median(residuos),
            "rmse": np.sqrt(np.mean(residuos ** 2)),
            "percentil_95_absoluto": np.percentile(np.abs(residuos), 95)
        })

df_resumen_residuos_trainseguro = pd.DataFrame(filas_resumen_residuos)

display(df_resumen_residuos_trainseguro)

,particion,variable,trayectorias,residuo_medio,residuo_mediano,rmse,percentil_95_absoluto
0,entrenamiento,Longitud SCE,12904,1.203738e-13,0.018864,0.395654,0.814368
1,entrenamiento,Latitud eclíptica,12904,2.276886e-15,0.011981,0.379410,0.805422
2,entrenamiento,Velocidad geocéntrica,12904,-2.385362e-15,-0.009842,0.600880,1.251636
3,prueba,Longitud SCE,6175,7.265455e-03,0.022427,0.386778,0.790141
4,prueba,Latitud eclíptica,6175,1.277238e-02,0.021254,0.373265,0.772744
5,prueba,Velocidad geocéntrica,6175,-1.108226e-02,-0.030084,0.581962,1.202630


## Construir y auditar variables de PCA

In [15]:
# Construir los vectores de velocidad observados y esperados en coordenadas cartesianas

def construir_vector_velocidad(lambda_deg, beta_deg, velocidad_km_s):
    lambda_rad = np.deg2rad(np.asarray(lambda_deg, dtype=float))
    beta_rad = np.deg2rad(np.asarray(beta_deg, dtype=float))
    velocidad_km_s = np.asarray(velocidad_km_s, dtype=float)

    componente_x = velocidad_km_s * np.cos(beta_rad) * np.cos(lambda_rad)
    componente_y = velocidad_km_s * np.cos(beta_rad) * np.sin(lambda_rad)
    componente_z = velocidad_km_s * np.sin(beta_rad)

    return np.column_stack([componente_x, componente_y, componente_z])

vectores_observados = construir_vector_velocidad(df_modelado_trainseguro["lambda_sce_deg"], df_modelado_trainseguro["BETgeo (deg)"], df_modelado_trainseguro["Vgeo (km/s)"])
vectores_esperados = construir_vector_velocidad(df_modelado_trainseguro["lambda_sce_esperada_train_deg"], df_modelado_trainseguro["beta_esperada_train_deg"], df_modelado_trainseguro["vgeo_esperada_train_km_s"])
vectores_residuales = vectores_observados - vectores_esperados

df_modelado_trainseguro[["delta_v_x_km_s", "delta_v_y_km_s", "delta_v_z_km_s"]] = vectores_residuales
df_modelado_trainseguro["norma_delta_v_km_s"] = np.linalg.norm(vectores_residuales, axis=1)

df_entrenamiento_trainseguro = df_modelado_trainseguro.loc[df_modelado_trainseguro["particion_modelado"] == "entrenamiento"].copy()
df_prueba_trainseguro = df_modelado_trainseguro.loc[df_modelado_trainseguro["particion_modelado"] == "prueba"].copy()

In [16]:
# Verificar matemáticamente los vectores de velocidad y sus residuos

normas_observadas = np.linalg.norm(vectores_observados, axis=1)
normas_esperadas = np.linalg.norm(vectores_esperados, axis=1)
normas_residuales = np.linalg.norm(vectores_residuales, axis=1)

filas_auditoria_vectores = [
    {
        "comprobacion": "Norma observada igual a Vgeo",
        "error_maximo": np.max(np.abs(normas_observadas - df_modelado_trainseguro["Vgeo (km/s)"].to_numpy(dtype=float)))
    },
    {
        "comprobacion": "Norma esperada igual a Vgeo esperada",
        "error_maximo": np.max(np.abs(normas_esperadas - df_modelado_trainseguro["vgeo_esperada_train_km_s"].to_numpy(dtype=float)))
    },
    {
        "comprobacion": "Norma residual almacenada correctamente",
        "error_maximo": np.max(np.abs(normas_residuales - df_modelado_trainseguro["norma_delta_v_km_s"].to_numpy(dtype=float)))
    }
]

df_auditoria_vectores = pd.DataFrame(filas_auditoria_vectores)

if not np.isfinite(vectores_residuales).all():
    raise ValueError("Los vectores residuales contienen valores no finitos.")

display(df_auditoria_vectores)

,comprobacion,error_maximo
0,Norma observada igual a Vgeo,7.105427e-15
1,Norma esperada igual a Vgeo esperada,1.421085e-14
2,Norma residual almacenada correctamente,0.000000e+00


In [17]:
# Resumir las componentes de velocidad residual en entrenamiento y prueba

columnas_delta_v = ["delta_v_x_km_s", "delta_v_y_km_s", "delta_v_z_km_s"]
filas_resumen_delta_v = []

for particion, df_particion in df_modelado_trainseguro.groupby("particion_modelado", observed=True):
    for columna in columnas_delta_v:
        valores = df_particion[columna].to_numpy(dtype=float)

        filas_resumen_delta_v.append({
            "particion": particion,
            "variable": columna,
            "trayectorias": len(valores),
            "media": np.mean(valores),
            "mediana": np.median(valores),
            "desviacion_estandar": np.std(valores, ddof=1)
        })

    normas = df_particion["norma_delta_v_km_s"].to_numpy(dtype=float)

    filas_resumen_delta_v.append({
        "particion": particion,
        "variable": "norma_delta_v_km_s",
        "trayectorias": len(normas),
        "media": np.mean(normas),
        "mediana": np.median(normas),
        "desviacion_estandar": np.std(normas, ddof=1)
    })

df_resumen_delta_v = pd.DataFrame(filas_resumen_delta_v)

display(df_resumen_delta_v)

,particion,variable,trayectorias,media,mediana,desviacion_estandar
0,entrenamiento,delta_v_x_km_s,12904,0.000615,0.018898,0.572376
1,entrenamiento,delta_v_y_km_s,12904,0.001991,-0.009037,0.280617
2,entrenamiento,delta_v_z_km_s,12904,-0.000365,0.008112,0.240339
3,entrenamiento,norma_delta_v_km_s,12904,0.548289,0.444859,0.404331
4,prueba,delta_v_x_km_s,6175,0.013336,0.039372,0.559901
5,prueba,delta_v_y_km_s,6175,0.004055,-0.006006,0.261957
6,prueba,delta_v_z_km_s,6175,0.005052,0.012651,0.235149
7,prueba,norma_delta_v_km_s,6175,0.535099,0.430014,0.388906


In [18]:
# Construir la base local de velocidad alrededor del radiante esperado de cada trayectoria

def construir_base_local_velocidad(lambda_centro_deg, beta_centro_deg):
    lambda_rad = np.deg2rad(np.asarray(lambda_centro_deg, dtype=float))
    beta_rad = np.deg2rad(np.asarray(beta_centro_deg, dtype=float))

    eje_paralelo = np.column_stack([
        np.cos(beta_rad) * np.cos(lambda_rad),
        np.cos(beta_rad) * np.sin(lambda_rad),
        np.sin(beta_rad)
    ])

    eje_lambda = np.column_stack([
        -np.sin(lambda_rad),
        np.cos(lambda_rad),
        np.zeros_like(lambda_rad)
    ])

    eje_beta = np.column_stack([
        -np.sin(beta_rad) * np.cos(lambda_rad),
        -np.sin(beta_rad) * np.sin(lambda_rad),
        np.cos(beta_rad)
    ])

    return eje_paralelo, eje_lambda, eje_beta

ejes_paralelos, ejes_lambda, ejes_beta = construir_base_local_velocidad(
    df_modelado_trainseguro["lambda_sce_esperada_train_deg"],
    df_modelado_trainseguro["beta_esperada_train_deg"]
)

primera_base_local = pd.DataFrame(
    [ejes_paralelos[0], ejes_lambda[0], ejes_beta[0]],
    index=["eje_paralelo", "eje_lambda", "eje_beta"],
    columns=["componente_x", "componente_y", "componente_z"]
)

print("Bases locales construidas:", len(ejes_paralelos))
print("Forma de cada conjunto de ejes:", ejes_paralelos.shape)
display(primera_base_local)

Bases locales construidas: 19079
Forma de cada conjunto de ejes: (19079, 3)


,componente_x,componente_y,componente_z
eje_paralelo,-0.863702,-0.469309,0.183763
eje_lambda,0.477439,-0.878665,0.000000
eje_beta,0.161466,0.087736,0.982971


In [19]:
# Proyectar cada vector residual sobre su base local paralela, longitudinal y latitudinal

df_modelado_trainseguro["delta_v_paralela_km_s"] = np.einsum("ij,ij->i", vectores_residuales, ejes_paralelos)
df_modelado_trainseguro["delta_v_lambda_km_s"] = np.einsum("ij,ij->i", vectores_residuales, ejes_lambda)
df_modelado_trainseguro["delta_v_beta_km_s"] = np.einsum("ij,ij->i", vectores_residuales, ejes_beta)

df_entrenamiento_trainseguro = df_modelado_trainseguro.loc[df_modelado_trainseguro["particion_modelado"] == "entrenamiento"].copy()
df_prueba_trainseguro = df_modelado_trainseguro.loc[df_modelado_trainseguro["particion_modelado"] == "prueba"].copy()

columnas_componentes_locales = [
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
    "norma_delta_v_km_s"
]

display(df_modelado_trainseguro[columnas_componentes_locales].head())

,delta_v_paralela_km_s,delta_v_lambda_km_s,delta_v_beta_km_s,norma_delta_v_km_s
0,0.010251,0.060752,-0.836403,0.838669
1,0.539424,-0.381939,-0.546822,0.857829
2,-1.219129,0.491805,0.577951,1.436028
3,0.491230,0.040037,0.056796,0.496120
4,0.027352,0.123178,0.523088,0.538091


In [20]:
# Auditar la base local, la proyección y la conservación del vector residual

componentes_locales = df_modelado_trainseguro[
    ["delta_v_paralela_km_s", "delta_v_lambda_km_s", "delta_v_beta_km_s"]
].to_numpy(dtype=float)

vectores_reconstruidos = (
    componentes_locales[:, 0, None] * ejes_paralelos
    + componentes_locales[:, 1, None] * ejes_lambda
    + componentes_locales[:, 2, None] * ejes_beta
)

normas_locales = np.linalg.norm(componentes_locales, axis=1)
velocidades_observadas = df_modelado_trainseguro["Vgeo (km/s)"].to_numpy(dtype=float)
velocidades_esperadas = df_modelado_trainseguro["vgeo_esperada_train_km_s"].to_numpy(dtype=float)
direcciones_observadas = vectores_observados / velocidades_observadas[:, None]
cosenos_angulares = np.clip(np.einsum("ij,ij->i", direcciones_observadas, ejes_paralelos), -1.0, 1.0)

paralela_teorica = velocidades_observadas * cosenos_angulares - velocidades_esperadas
transversal_local = np.sqrt(componentes_locales[:, 1] ** 2 + componentes_locales[:, 2] ** 2)
transversal_teorica = velocidades_observadas * np.sqrt(np.maximum(0.0, 1.0 - cosenos_angulares ** 2))

filas_auditoria_local = [
    {
        "comprobacion": "Norma del eje paralelo",
        "error_maximo": np.max(np.abs(np.linalg.norm(ejes_paralelos, axis=1) - 1)),
        "unidad": "sin unidad"
    },
    {
        "comprobacion": "Norma del eje lambda",
        "error_maximo": np.max(np.abs(np.linalg.norm(ejes_lambda, axis=1) - 1)),
        "unidad": "sin unidad"
    },
    {
        "comprobacion": "Norma del eje beta",
        "error_maximo": np.max(np.abs(np.linalg.norm(ejes_beta, axis=1) - 1)),
        "unidad": "sin unidad"
    },
    {
        "comprobacion": "Paralelo · lambda",
        "error_maximo": np.max(np.abs(np.einsum("ij,ij->i", ejes_paralelos, ejes_lambda))),
        "unidad": "sin unidad"
    },
    {
        "comprobacion": "Paralelo · beta",
        "error_maximo": np.max(np.abs(np.einsum("ij,ij->i", ejes_paralelos, ejes_beta))),
        "unidad": "sin unidad"
    },
    {
        "comprobacion": "Lambda · beta",
        "error_maximo": np.max(np.abs(np.einsum("ij,ij->i", ejes_lambda, ejes_beta))),
        "unidad": "sin unidad"
    },
    {
        "comprobacion": "Orientación lambda × beta = paralelo",
        "error_maximo": np.max(np.linalg.norm(np.cross(ejes_lambda, ejes_beta) - ejes_paralelos, axis=1)),
        "unidad": "sin unidad"
    },
    {
        "comprobacion": "Reconstrucción del vector residual",
        "error_maximo": np.max(np.linalg.norm(vectores_reconstruidos - vectores_residuales, axis=1)),
        "unidad": "km/s"
    },
    {
        "comprobacion": "Conservación de la norma residual",
        "error_maximo": np.max(np.abs(normas_locales - df_modelado_trainseguro["norma_delta_v_km_s"].to_numpy(dtype=float))),
        "unidad": "km/s"
    },
    {
        "comprobacion": "Identidad de la componente paralela",
        "error_maximo": np.max(np.abs(componentes_locales[:, 0] - paralela_teorica)),
        "unidad": "km/s"
    },
    {
        "comprobacion": "Identidad de la componente transversal",
        "error_maximo": np.max(np.abs(transversal_local - transversal_teorica)),
        "unidad": "km/s"
    }
]

df_auditoria_componentes_locales = pd.DataFrame(filas_auditoria_local)
cantidad_no_finitos_locales = int(componentes_locales.size - np.isfinite(componentes_locales).sum())

if cantidad_no_finitos_locales > 0:
    raise ValueError(f"Las componentes locales contienen {cantidad_no_finitos_locales} valores no finitos.")

if df_auditoria_componentes_locales["error_maximo"].max() > 1e-9:
    raise ValueError("La descomposición local no superó la tolerancia numérica establecida.")

display(df_auditoria_componentes_locales)
print("Valores no finitos:", cantidad_no_finitos_locales)
print("La descomposición física local superó todas las pruebas.")

,comprobacion,error_maximo,unidad
0,Norma del eje paralelo,2.220446e-16,sin unidad
1,Norma del eje lambda,1.110223e-16,sin unidad
2,Norma del eje beta,1.110223e-16,sin unidad
3,Paralelo · lambda,5.551115e-17,sin unidad
4,Paralelo · beta,6.245005e-17,sin unidad
5,Lambda · beta,1.387779e-17,sin unidad
6,Orientación lambda × beta = paralelo,5.551115e-17,sin unidad
7,Reconstrucción del vector residual,9.992007e-16,km/s
8,Conservación de la norma residual,8.881784e-16,km/s
9,Identidad de la componente paralela,1.676437e-14,km/s


Valores no finitos: 0
La descomposición física local superó todas las pruebas.


In [21]:
# Resumir la distribución de las componentes locales en entrenamiento y prueba

columnas_resumen_local = [
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
    "norma_delta_v_km_s"
]

filas_resumen_componentes_locales = []

for particion, df_particion in df_modelado_trainseguro.groupby("particion_modelado", observed=True):
    for columna in columnas_resumen_local:
        valores = df_particion[columna].to_numpy(dtype=float)

        filas_resumen_componentes_locales.append({
            "particion": particion,
            "variable": columna,
            "trayectorias": len(valores),
            "minimo": np.min(valores),
            "percentil_5": np.percentile(valores, 5),
            "percentil_25": np.percentile(valores, 25),
            "mediana": np.median(valores),
            "media": np.mean(valores),
            "percentil_75": np.percentile(valores, 75),
            "percentil_95": np.percentile(valores, 95),
            "maximo": np.max(valores),
            "desviacion_estandar": np.std(valores, ddof=1),
            "percentil_95_absoluto": np.percentile(np.abs(valores), 95)
        })

df_resumen_componentes_locales = pd.DataFrame(filas_resumen_componentes_locales)

display(df_resumen_componentes_locales.round(6))

,particion,variable,trayectorias,minimo,percentil_5,percentil_25,mediana,media,percentil_75,percentil_95,maximo,desviacion_estandar,percentil_95_absoluto
0,entrenamiento,delta_v_paralela_km_s,12904,-3.478542,-0.948664,-0.316536,-0.010648,-0.001523,0.309563,0.961607,3.471391,0.600805,1.250566
1,entrenamiento,delta_v_lambda_km_s,12904,-1.245482,-0.389413,-0.129664,0.010952,-0.001467,0.138474,0.353037,1.192395,0.230229,0.470412
2,entrenamiento,delta_v_beta_km_s,12904,-1.179222,-0.382217,-0.117172,0.007188,-0.000080,0.125078,0.354662,1.108763,0.223944,0.474382
3,entrenamiento,norma_delta_v_km_s,12904,0.010936,0.131588,0.272920,0.444859,0.548289,0.701917,1.344216,3.480993,0.404331,1.344216
4,prueba,delta_v_paralela_km_s,6175,-3.390972,-0.914875,-0.331258,-0.031742,-0.012544,0.278610,0.959533,3.348503,0.581830,1.201258
5,prueba,delta_v_lambda_km_s,6175,-1.172619,-0.375163,-0.124917,0.012786,0.002664,0.145915,0.347631,0.782803,0.224859,0.456549
6,prueba,delta_v_beta_km_s,6175,-1.082213,-0.354607,-0.107814,0.012670,0.007476,0.130210,0.352083,1.102348,0.219812,0.457769
7,prueba,norma_delta_v_km_s,6175,0.013597,0.128764,0.269482,0.430014,0.535099,0.684177,1.281410,3.490513,0.388906,1.281410


## Preparar estaciones, brillo y partición

In [22]:
# Inspeccionar el formato de la columna que contiene las estaciones participantes

frecuencias_estaciones = (
    df_modelado_trainseguro["Participating (stations)"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("estaciones_registradas")
    .reset_index(name="repeticiones")
)

display(frecuencias_estaciones.head(25))

,estaciones_registradas,repeticiones
0,"ES0007,ES0008",164
1,"US002D,US002L",108
2,"ES000U,PT0002",107
3,"US000S,US001E",87
4,"ES001A,PT0002",79
5,"USV001,USV002",66
6,"AU001K,AU001L",65
7,"CA000S,CA000V",57
8,"USL017,USL01B",56
9,"US001V,US0051",56


In [23]:
# Contar las estaciones participantes registradas para cada trayectoria

df_modelado_trainseguro["numero_estaciones"] = df_modelado_trainseguro["Participating (stations)"].astype("string").str.split(",").str.len().astype("Int64")

resumen_numero_estaciones = pd.DataFrame({
    "comprobacion": ["Valores faltantes", "Mínimo", "Mediana", "Máximo"],
    "resultado": [
        int(df_modelado_trainseguro["numero_estaciones"].isna().sum()),
        int(df_modelado_trainseguro["numero_estaciones"].min()),
        float(df_modelado_trainseguro["numero_estaciones"].median()),
        int(df_modelado_trainseguro["numero_estaciones"].max())
    ]
})

display(resumen_numero_estaciones)
display(df_modelado_trainseguro["numero_estaciones"].value_counts().sort_index().rename_axis("numero_estaciones").reset_index(name="trayectorias"))

,comprobacion,resultado
0,Valores faltantes,0.0
1,Mínimo,2.0
2,Mediana,3.0
3,Máximo,22.0


,numero_estaciones,trayectorias
0,2,8027
1,3,4012
2,4,2436
3,5,1546
4,6,902
5,7,544
6,8,404
7,9,321
8,10,253
9,11,179


In [24]:
# Auditar las variables observacionales que se utilizarán como controles del modelo

columnas_calidad_ml = ["Qc (deg)", "MedianFitErr (arcsec)", "numero_estaciones", "sigma_radiante_aprox", "Vgeo_sigma"]
filas_auditoria_calidad = []

for columna in columnas_calidad_ml:
    valores = pd.to_numeric(df_modelado_trainseguro[columna], errors="coerce")

    filas_auditoria_calidad.append({
        "variable": columna,
        "trayectorias": len(valores),
        "valores_faltantes": int(valores.isna().sum()),
        "valores_no_finitos": int((~np.isfinite(valores.dropna())).sum()),
        "valores_iguales_a_cero": int((valores == 0).sum()),
        "minimo": float(valores.min()),
        "percentil_25": float(valores.quantile(0.25)),
        "mediana": float(valores.median()),
        "percentil_75": float(valores.quantile(0.75)),
        "maximo": float(valores.max())
    })

df_auditoria_calidad_ml = pd.DataFrame(filas_auditoria_calidad)

display(df_auditoria_calidad_ml)

,variable,trayectorias,valores_faltantes,valores_no_finitos,valores_iguales_a_cero,minimo,percentil_25,mediana,percentil_75,maximo
0,Qc (deg),19079,0,0,0,15.060000,46.750000,67.160000,82.190000,90.000000
1,MedianFitErr (arcsec),19079,0,0,0,2.730000,34.940000,43.620000,51.310000,60.000000
2,numero_estaciones,19079,0,0,0,2.000000,2.000000,3.000000,4.000000,22.000000
3,sigma_radiante_aprox,19079,0,0,0,0.009896,0.113827,0.161493,0.204763,0.249978
4,Vgeo_sigma,19079,0,0,26,0.000000,0.027600,0.059000,0.112700,2.273700


In [25]:
# Preparar la incertidumbre de velocidad para el modelado sin modificar la variable original

mascara_entrenamiento = df_modelado_trainseguro["particion_modelado"] == "entrenamiento"
mascara_vgeo_sigma_cero = df_modelado_trainseguro["Vgeo_sigma"] == 0
mediana_vgeo_sigma_train = df_modelado_trainseguro.loc[mascara_entrenamiento & ~mascara_vgeo_sigma_cero, "Vgeo_sigma"].median()

if not (mascara_vgeo_sigma_cero == df_modelado_trainseguro["flag_vgeo_sigma_cero"].astype(bool)).all():
    raise ValueError("El indicador existente de Vgeo_sigma igual a cero no coincide con los valores observados.")

df_modelado_trainseguro["Vgeo_sigma_modelo"] = df_modelado_trainseguro["Vgeo_sigma"].mask(mascara_vgeo_sigma_cero, mediana_vgeo_sigma_train)

df_entrenamiento_trainseguro = df_modelado_trainseguro.loc[mascara_entrenamiento].copy()
df_prueba_trainseguro = df_modelado_trainseguro.loc[~mascara_entrenamiento].copy()

df_resumen_imputacion_vgeo = (
    df_modelado_trainseguro
    .groupby("particion_modelado", observed=True)
    .agg(
        trayectorias=("id_fila_origen", "size"),
        valores_cero_originales=("flag_vgeo_sigma_cero", "sum"),
        minimo_original=("Vgeo_sigma", "min"),
        minimo_para_modelo=("Vgeo_sigma_modelo", "min"),
        mediana_para_modelo=("Vgeo_sigma_modelo", "median")
    )
    .reset_index()
)

print("Mediana positiva aprendida con entrenamiento:", mediana_vgeo_sigma_train)
display(df_resumen_imputacion_vgeo)

Mediana positiva aprendida con entrenamiento: 0.0593


,particion_modelado,trayectorias,valores_cero_originales,minimo_original,minimo_para_modelo,mediana_para_modelo
0,entrenamiento,12904,18,0.0,0.0005,0.0593
1,prueba,6175,8,0.0,0.0008,0.0589


In [26]:
# Definir los grupos de brillo usando exclusivamente la distribución de entrenamiento

mascara_fotometria_train = mascara_entrenamiento & df_modelado_trainseguro["fotometria_utilizable"]
magnitudes_train = df_modelado_trainseguro.loc[mascara_fotometria_train, "magnitud_analisis"].dropna()

limite_brillante_train = magnitudes_train.quantile(1 / 3)
limite_tenue_train = magnitudes_train.quantile(2 / 3)

mascara_magnitud_valida = df_modelado_trainseguro["fotometria_utilizable"] & df_modelado_trainseguro["magnitud_analisis"].notna()
mascara_brillante = mascara_magnitud_valida & (df_modelado_trainseguro["magnitud_analisis"] <= limite_brillante_train)
mascara_tenue = mascara_magnitud_valida & (df_modelado_trainseguro["magnitud_analisis"] >= limite_tenue_train)
mascara_intermedia = mascara_magnitud_valida & ~mascara_brillante & ~mascara_tenue

df_modelado_trainseguro["grupo_brillo_ml"] = pd.Series(pd.NA, index=df_modelado_trainseguro.index, dtype="string")
df_modelado_trainseguro.loc[mascara_brillante, "grupo_brillo_ml"] = "brillante"
df_modelado_trainseguro.loc[mascara_intermedia, "grupo_brillo_ml"] = "intermedio"
df_modelado_trainseguro.loc[mascara_tenue, "grupo_brillo_ml"] = "tenue"

df_modelado_trainseguro["objetivo_ml"] = pd.Series(pd.NA, index=df_modelado_trainseguro.index, dtype="Int64")
df_modelado_trainseguro.loc[mascara_brillante, "objetivo_ml"] = 0
df_modelado_trainseguro.loc[mascara_tenue, "objetivo_ml"] = 1
df_modelado_trainseguro["usar_modelo_ml"] = df_modelado_trainseguro["objetivo_ml"].notna()

df_entrenamiento_trainseguro = df_modelado_trainseguro.loc[mascara_entrenamiento].copy()
df_prueba_trainseguro = df_modelado_trainseguro.loc[~mascara_entrenamiento].copy()

print("Límite de meteoros brillantes:", limite_brillante_train)
print("Límite de meteoros tenues:", limite_tenue_train)

Límite de meteoros brillantes: -0.89
Límite de meteoros tenues: -0.07


In [27]:
# Auditar los grupos de brillo en las particiones estándar y estricta

filas_resumen_grupos_ml = []

for particion in ["entrenamiento", "prueba"]:
    df_particion = df_modelado_trainseguro.loc[df_modelado_trainseguro["particion_modelado"] == particion]

    for muestra, df_muestra in {
        "Estándar": df_particion,
        "Estricta": df_particion.loc[df_particion["pertenece_muestra_estricta"]]
    }.items():
        conteos = df_muestra["grupo_brillo_ml"].value_counts(dropna=False)

        filas_resumen_grupos_ml.append({
            "particion": particion,
            "muestra": muestra,
            "trayectorias": len(df_muestra),
            "brillantes": int(conteos.get("brillante", 0)),
            "intermedias": int(conteos.get("intermedio", 0)),
            "tenues": int(conteos.get("tenue", 0)),
            "fotometria_no_utilizable": int(df_muestra["grupo_brillo_ml"].isna().sum()),
            "utilizables_ml": int(df_muestra["usar_modelo_ml"].sum())
        })

df_resumen_grupos_ml = pd.DataFrame(filas_resumen_grupos_ml)

display(df_resumen_grupos_ml)

,particion,muestra,trayectorias,brillantes,intermedias,tenues,fotometria_no_utilizable,utilizables_ml
0,entrenamiento,Estándar,12904,4300,4247,4332,25,8632
1,entrenamiento,Estricta,4166,1681,1418,1046,21,2727
2,prueba,Estándar,6175,2133,1993,2049,0,4182
3,prueba,Estricta,2874,1191,931,752,0,1943


## Crear base, diccionario y parámetros

In [28]:
# Construir la base final incorporando las componentes físicas locales para PCA

columnas_pca = ["delta_v_paralela_km_s", "delta_v_lambda_km_s", "delta_v_beta_km_s"]
columnas_calidad_modelo = ["Qc (deg)", "MedianFitErr (arcsec)", "numero_estaciones", "sigma_radiante_aprox", "Vgeo_sigma_modelo"]

df_modelado_trainseguro["usar_pca"] = np.isfinite(df_modelado_trainseguro[columnas_pca]).all(axis=1)

columnas_base_modelado = [
    "id_fila_origen",
    "Beginning (UTC Time)",
    "source_year",
    "particion_modelado",
    "pertenece_muestra_estricta",
    "nivel_seleccion",
    "fotometria_utilizable",
    "Peak (AbsMag)",
    "magnitud_analisis",
    "flag_fotometria_sospechosa",
    "grupo_brillo_ml",
    "objetivo_ml",
    "usar_modelo_ml",
    "usar_pca",
    "delta_sol_lon_deg",
    "lambda_sce_deg",
    "BETgeo (deg)",
    "Vgeo (km/s)",
    "lambda_sce_esperada_train_deg",
    "beta_esperada_train_deg",
    "vgeo_esperada_train_km_s",
    "residuo_lambda_train_deg",
    "residuo_beta_train_deg",
    "residuo_vgeo_train_km_s",
    "delta_v_x_km_s",
    "delta_v_y_km_s",
    "delta_v_z_km_s",
    "delta_v_paralela_km_s",
    "delta_v_lambda_km_s",
    "delta_v_beta_km_s",
    "norma_delta_v_km_s",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Participating (stations)",
    "numero_estaciones",
    "sigma_radiante_aprox",
    "Vgeo_sigma",
    "flag_vgeo_sigma_cero",
    "Vgeo_sigma_modelo"
]

df_base_modelado_final = df_modelado_trainseguro[columnas_base_modelado].copy()
df_modelado_entrenamiento = df_base_modelado_final.loc[df_base_modelado_final["particion_modelado"] == "entrenamiento"].copy()
df_modelado_prueba = df_base_modelado_final.loc[df_base_modelado_final["particion_modelado"] == "prueba"].copy()

filas_auditoria_base_modelado = [
    {
        "comprobacion": "Filas totales",
        "resultado": len(df_base_modelado_final)
    },
    {
        "comprobacion": "Filas de entrenamiento",
        "resultado": len(df_modelado_entrenamiento)
    },
    {
        "comprobacion": "Filas de prueba",
        "resultado": len(df_modelado_prueba)
    },
    {
        "comprobacion": "Filas utilizables para PCA",
        "resultado": int(df_base_modelado_final["usar_pca"].sum())
    },
    {
        "comprobacion": "Faltantes en variables PCA",
        "resultado": int(df_base_modelado_final[columnas_pca].isna().sum().sum())
    },
    {
        "comprobacion": "Faltantes en controles de calidad",
        "resultado": int(df_base_modelado_final[columnas_calidad_modelo].isna().sum().sum())
    },
    {
        "comprobacion": "Objetivos faltantes entre filas utilizables para ML",
        "resultado": int(df_base_modelado_final.loc[df_base_modelado_final["usar_modelo_ml"], "objetivo_ml"].isna().sum())
    }
]

df_auditoria_base_modelado = pd.DataFrame(filas_auditoria_base_modelado)

display(df_auditoria_base_modelado)
print("Variables definitivas del PCA:", columnas_pca)
print("Dimensión de la base final:", df_base_modelado_final.shape)

,comprobacion,resultado
0,Filas totales,19079
1,Filas de entrenamiento,12904
2,Filas de prueba,6175
3,Filas utilizables para PCA,19079
4,Faltantes en variables PCA,0
5,Faltantes en controles de calidad,0
6,Objetivos faltantes entre filas utilizables pa...,0


Variables definitivas del PCA: ['delta_v_paralela_km_s', 'delta_v_lambda_km_s', 'delta_v_beta_km_s']
Dimensión de la base final: (19079, 39)


In [29]:
# Inspeccionar la estructura del diccionario físico para extenderlo con las variables de modelado

print("Columnas del diccionario:", df_diccionario_fisico.columns.tolist())
display(df_diccionario_fisico.head(10))

Columnas del diccionario: ['variable', 'categoria', 'tipo_dato', 'unidad', 'descripcion', 'valores_faltantes', 'valores_unicos']


,variable,categoria,tipo_dato,unidad,descripcion,valores_faltantes,valores_unicos
0,id_fila_origen,Trazabilidad,int64,sin unidad,Índice de la trayectoria en la base procesada ...,0,19079
1,pertenece_muestra_estandar,Selección,bool,booleano,Indica si la trayectoria pertenece a la muestr...,0,1
2,pertenece_muestra_estricta,Selección,bool,booleano,Indica si también satisface los criterios de l...,0,2
3,nivel_seleccion,Selección,str,categoría,Nivel máximo de selección satisfecho por la tr...,0,2
4,fotometria_utilizable,Fotometría,bool,booleano,Indica si la magnitud puede utilizarse en el a...,0,2
5,Beginning (Julian date),Tiempo,float64,día juliano,Instante inicial de la trayectoria expresado c...,0,19079
6,Beginning (UTC Time),Tiempo,"datetime64[us, UTC]",UTC,Instante inicial de la trayectoria en tiempo u...,0,19079
7,source_year,Tiempo,int64,año,Año calendario utilizado para los ajustes e in...,0,7
8,IAU (No),Clasificación,int64,sin unidad,Número IAU asignado a la lluvia de meteoros.,0,1
9,IAU (code),Clasificación,string,código,Código IAU de la lluvia; GEM para las Gemínidas.,0,1


In [30]:
# Extender el diccionario físico con las componentes locales y las demás variables de modelado

documentacion_nueva = {
    "particion_modelado": {"categoria": "Partición", "unidad": "categoría", "descripcion": "Indica si la trayectoria pertenece al entrenamiento de 2019–2024 o a la prueba temporal independiente de 2025."},
    "grupo_brillo_ml": {"categoria": "Objetivo ML", "unidad": "categoría", "descripcion": "Grupo de brillo definido con los terciles de magnitud del entrenamiento: brillante, intermedio o tenue; queda vacío cuando la fotometría no es utilizable."},
    "objetivo_ml": {"categoria": "Objetivo ML", "unidad": "clase binaria", "descripcion": "Variable objetivo del clasificador: 0 para meteoro brillante y 1 para meteoro tenue; queda vacía para casos intermedios o sin fotometría utilizable."},
    "usar_modelo_ml": {"categoria": "Objetivo ML", "unidad": "booleano", "descripcion": "Indica si la trayectoria pertenece a uno de los dos grupos extremos de brillo y puede utilizarse en la clasificación binaria."},
    "usar_pca": {"categoria": "PCA", "unidad": "booleano", "descripcion": "Indica si las componentes paralela, longitudinal y latitudinal de la velocidad residual están disponibles y son finitas para el PCA."},
    "lambda_sce_esperada_train_deg": {"categoria": "Centro físico de entrenamiento", "unidad": "grados", "descripcion": "Longitud eclíptica centrada en el Sol esperada según el modelo lineal ajustado exclusivamente con 2019–2024."},
    "beta_esperada_train_deg": {"categoria": "Centro físico de entrenamiento", "unidad": "grados", "descripcion": "Latitud eclíptica esperada según el modelo cuadrático ajustado exclusivamente con 2019–2024."},
    "vgeo_esperada_train_km_s": {"categoria": "Centro físico de entrenamiento", "unidad": "km/s", "descripcion": "Velocidad geocéntrica esperada según el modelo lineal ajustado exclusivamente con 2019–2024."},
    "residuo_lambda_train_deg": {"categoria": "Residuo físico", "unidad": "grados", "descripcion": "Diferencia angular envuelta entre la longitud SCE observada y la esperada por el modelo de entrenamiento."},
    "residuo_beta_train_deg": {"categoria": "Residuo físico", "unidad": "grados", "descripcion": "Diferencia entre la latitud eclíptica observada y la esperada por el modelo de entrenamiento."},
    "residuo_vgeo_train_km_s": {"categoria": "Residuo físico", "unidad": "km/s", "descripcion": "Diferencia entre la velocidad geocéntrica observada y la esperada por el modelo de entrenamiento."},
    "delta_v_x_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Componente x del vector de velocidad observado menos el esperado, expresado en los ejes cartesianos del sistema eclíptico centrado en el Sol."},
    "delta_v_y_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Componente y del vector de velocidad observado menos el esperado, expresado en los ejes cartesianos del sistema eclíptico centrado en el Sol."},
    "delta_v_z_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Componente z del vector de velocidad observado menos el esperado, expresado en los ejes cartesianos del sistema eclíptico centrado en el Sol."},
    "delta_v_paralela_km_s": {"categoria": "PCA", "unidad": "km/s", "descripcion": "Proyección del vector de velocidad residual sobre la dirección del radiante esperado; un valor positivo indica una componente observada mayor en esa dirección."},
    "delta_v_lambda_km_s": {"categoria": "PCA", "unidad": "km/s", "descripcion": "Proyección del vector de velocidad residual sobre la dirección tangente local de longitud eclíptica centrada en el Sol creciente."},
    "delta_v_beta_km_s": {"categoria": "PCA", "unidad": "km/s", "descripcion": "Proyección del vector de velocidad residual sobre la dirección tangente local de latitud eclíptica creciente."},
    "norma_delta_v_km_s": {"categoria": "Velocidad residual", "unidad": "km/s", "descripcion": "Norma euclidiana del vector tridimensional de velocidad observado menos el esperado; es invariante ante el cambio entre bases cartesianas y locales."},
    "numero_estaciones": {"categoria": "Calidad observacional", "unidad": "conteo", "descripcion": "Número de identificadores únicos de estaciones participantes registrados para la trayectoria."},
    "Vgeo_sigma_modelo": {"categoria": "Calidad observacional", "unidad": "km/s", "descripcion": "Incertidumbre de velocidad utilizada por el modelo; los valores originales iguales a cero fueron sustituidos por la mediana positiva del entrenamiento."}
}

variables_documentadas_originales = set(df_diccionario_fisico["variable"])
variables_documentadas_nuevas = set(documentacion_nueva)
variables_sin_documentacion = set(columnas_base_modelado) - variables_documentadas_originales - variables_documentadas_nuevas

if variables_sin_documentacion:
    raise ValueError(f"Faltan descripciones para: {sorted(variables_sin_documentacion)}")

df_diccionario_modelado = df_diccionario_fisico.loc[df_diccionario_fisico["variable"].isin(columnas_base_modelado)].copy()
filas_documentacion_nueva = []

for variable, documentacion in documentacion_nueva.items():
    serie = df_base_modelado_final[variable]

    filas_documentacion_nueva.append({
        "variable": variable,
        "categoria": documentacion["categoria"],
        "tipo_dato": str(serie.dtype),
        "unidad": documentacion["unidad"],
        "descripcion": documentacion["descripcion"],
        "valores_faltantes": int(serie.isna().sum()),
        "valores_unicos": int(serie.nunique(dropna=True))
    })

df_diccionario_modelado = pd.concat([df_diccionario_modelado, pd.DataFrame(filas_documentacion_nueva)], ignore_index=True)
orden_columnas_modelado = {variable: posicion for posicion, variable in enumerate(columnas_base_modelado, start=1)}
df_diccionario_modelado["posicion"] = df_diccionario_modelado["variable"].map(orden_columnas_modelado)
df_diccionario_modelado = df_diccionario_modelado.sort_values("posicion").reset_index(drop=True)
df_diccionario_modelado = df_diccionario_modelado[["posicion", "variable", "categoria", "tipo_dato", "unidad", "descripcion", "valores_faltantes", "valores_unicos"]]

if len(df_diccionario_modelado) != len(columnas_base_modelado):
    raise ValueError("El diccionario final no contiene exactamente una fila por variable.")

display(df_diccionario_modelado)
print("Variables de modelado:", len(columnas_base_modelado))
print("Variables documentadas:", len(df_diccionario_modelado))

,posicion,variable,categoria,tipo_dato,unidad,descripcion,valores_faltantes,valores_unicos
0,1,id_fila_origen,Trazabilidad,int64,sin unidad,Índice de la trayectoria en la base procesada ...,0,19079
1,2,Beginning (UTC Time),Tiempo,"datetime64[us, UTC]",UTC,Instante inicial de la trayectoria en tiempo u...,0,19079
2,3,source_year,Tiempo,int64,año,Año calendario utilizado para los ajustes e in...,0,7
3,4,particion_modelado,Partición,str,categoría,Indica si la trayectoria pertenece al entrenam...,0,2
4,5,pertenece_muestra_estricta,Selección,bool,booleano,Indica si también satisface los criterios de l...,0,2
5,6,nivel_seleccion,Selección,str,categoría,Nivel máximo de selección satisfecho por la tr...,0,2
6,7,fotometria_utilizable,Fotometría,bool,booleano,Indica si la magnitud puede utilizarse en el a...,0,2
7,8,Peak (AbsMag),Fotometría,float64,magnitudes,Magnitud absoluta original del máximo de brillo.,0,664
8,9,magnitud_analisis,Fotometría,float64,magnitudes,Magnitud utilizada en los análisis; es NaN cua...,25,640
9,10,flag_fotometria_sospechosa,Fotometría,bool,booleano,Marca magnitudes extremadamente negativas cons...,0,2


Variables de modelado: 39
Variables documentadas: 39


In [31]:
# Crear la configuración metodológica actualizada y resumir las particiones de la base

df_configuracion_modelado = pd.DataFrame([
    {"elemento": "Versión", "valor": "geminidas_modelado_v2", "descripcion": "Segunda versión de la base de modelado, con la velocidad residual expresada en componentes físicas locales."},
    {"elemento": "Años de entrenamiento", "valor": "2019–2024", "descripcion": "Años permitidos para ajustar correcciones, PCA, escaladores y modelos."},
    {"elemento": "Año de prueba", "valor": "2025", "descripcion": "Prueba temporal independiente; no debe utilizarse para ajustar ni seleccionar modelos."},
    {"elemento": "Modelo de longitud", "valor": "Lineal común", "descripcion": "Ajustado con 2019–2024 mediante lambda_sce_deg en función de delta_sol_lon_deg."},
    {"elemento": "Modelo de latitud", "valor": "Cuadrático común", "descripcion": "Ajustado con 2019–2024 mediante BETgeo (deg) en función de delta_sol_lon_deg."},
    {"elemento": "Modelo de velocidad", "valor": "Lineal común", "descripcion": "Ajustado con 2019–2024 mediante Vgeo (km/s) en función de delta_sol_lon_deg."},
    {"elemento": "Convención del vector residual", "valor": "velocidad observada menos velocidad esperada", "descripcion": "La velocidad residual se define como delta_v = v_observada - v_esperada, usando vectores orientados hacia el radiante."},
    {"elemento": "Base física local", "valor": "paralela, lambda, beta", "descripcion": "Cada velocidad residual se proyecta sobre la dirección del radiante esperado y las direcciones tangentes locales de longitud y latitud crecientes."},
    {"elemento": "Variables del PCA", "valor": ", ".join(columnas_pca), "descripcion": "Componentes paralela, longitudinal y latitudinal de la velocidad residual expresadas en km/s."},
    {"elemento": "Ajuste del PCA", "valor": "Solo entrenamiento", "descripcion": "Los ejes principales se aprenden con 2019–2024 y después se aplican sin reajuste a 2025."},
    {"elemento": "Escalamiento del PCA", "valor": "Centrar sin estandarizar", "descripcion": "Las tres componentes comparten unidades de km/s y se desea conservar su varianza física relativa."},
    {"elemento": "Límite brillante", "valor": limite_brillante_train, "descripcion": "Magnitud máxima para la clase brillante, calculada con el tercil inferior del entrenamiento."},
    {"elemento": "Límite tenue", "valor": limite_tenue_train, "descripcion": "Magnitud mínima para la clase tenue, calculada con el tercil superior del entrenamiento."},
    {"elemento": "Codificación del objetivo", "valor": "0 = brillante; 1 = tenue", "descripcion": "Los casos intermedios y las fotometrías no utilizables se excluyen únicamente del clasificador."},
    {"elemento": "Controles de calidad", "valor": ", ".join(columnas_calidad_modelo), "descripcion": "Variables observacionales utilizadas por el modelo logístico de referencia."},
    {"elemento": "Tratamiento de Vgeo_sigma igual a cero", "valor": mediana_vgeo_sigma_train, "descripcion": "Sustitución por la mediana positiva calculada exclusivamente con entrenamiento."},
    {"elemento": "Muestra estándar", "valor": "Análisis principal", "descripcion": "Muestra completa utilizada para el resultado principal."},
    {"elemento": "Muestra estricta", "valor": "Análisis de sensibilidad", "descripcion": "Subconjunto de mayor calidad identificado mediante pertenece_muestra_estricta."}
])

filas_resumen_particiones = []

for particion, df_particion in df_base_modelado_final.groupby("particion_modelado", observed=True):
    filas_resumen_particiones.append({
        "particion": particion,
        "anios": ", ".join(map(str, sorted(df_particion["source_year"].unique()))),
        "trayectorias": len(df_particion),
        "utilizables_pca": int(df_particion["usar_pca"].sum()),
        "utilizables_ml": int(df_particion["usar_modelo_ml"].sum()),
        "trayectorias_estrictas": int(df_particion["pertenece_muestra_estricta"].sum()),
        "utilizables_ml_estrictas": int((df_particion["usar_modelo_ml"] & df_particion["pertenece_muestra_estricta"]).sum())
    })

df_resumen_particiones_modelado = pd.DataFrame(filas_resumen_particiones)

display(df_configuracion_modelado)
display(df_resumen_particiones_modelado)

print("Elementos de configuración:", len(df_configuracion_modelado))

,elemento,valor,descripcion
0,Versión,geminidas_modelado_v2,"Segunda versión de la base de modelado, con la..."
1,Años de entrenamiento,2019–2024,"Años permitidos para ajustar correcciones, PCA..."
2,Año de prueba,2025,Prueba temporal independiente; no debe utiliza...
3,Modelo de longitud,Lineal común,Ajustado con 2019–2024 mediante lambda_sce_deg...
4,Modelo de latitud,Cuadrático común,Ajustado con 2019–2024 mediante BETgeo (deg) e...
5,Modelo de velocidad,Lineal común,Ajustado con 2019–2024 mediante Vgeo (km/s) en...
6,Convención del vector residual,velocidad observada menos velocidad esperada,La velocidad residual se define como delta_v =...
7,Base física local,"paralela, lambda, beta",Cada velocidad residual se proyecta sobre la d...
8,Variables del PCA,"delta_v_paralela_km_s, delta_v_lambda_km_s, de...","Componentes paralela, longitudinal y latitudin..."
9,Ajuste del PCA,Solo entrenamiento,Los ejes principales se aprenden con 2019–2024...


,particion,anios,trayectorias,utilizables_pca,utilizables_ml,trayectorias_estrictas,utilizables_ml_estrictas
0,entrenamiento,"2019, 2020, 2021, 2022, 2023, 2024",12904,12904,8632,4166,2727
1,prueba,2025,6175,6175,4182,2874,1943


Elementos de configuración: 18


## Guardar bases y metadatos

In [32]:
# Guardar la segunda versión de las bases de modelado y todos sus metadatos

carpeta_metadata_modelado = carpeta_modelado / "metadata"
carpeta_modelado.mkdir(parents=True, exist_ok=True)
carpeta_metadata_modelado.mkdir(parents=True, exist_ok=True)

ruta_modelado_completo_parquet = carpeta_modelado / "geminidas_modelado_completo_v2.parquet"
ruta_entrenamiento_parquet = carpeta_modelado / "geminidas_modelado_entrenamiento_2019_2024_v2.parquet"
ruta_entrenamiento_csv = carpeta_modelado / "geminidas_modelado_entrenamiento_2019_2024_v2.csv"
ruta_prueba_parquet = carpeta_modelado / "geminidas_modelado_prueba_2025_v2.parquet"
ruta_prueba_csv = carpeta_modelado / "geminidas_modelado_prueba_2025_v2.csv"
ruta_diccionario_modelado = carpeta_metadata_modelado / "diccionario_variables_modelado_v2.csv"
ruta_configuracion_modelado = carpeta_metadata_modelado / "configuracion_modelado_v2.csv"
ruta_parametros_centro = carpeta_metadata_modelado / "parametros_centro_entrenamiento_v2.csv"
ruta_resumen_particiones = carpeta_metadata_modelado / "resumen_particiones_modelado_v2.csv"
ruta_auditoria_base = carpeta_metadata_modelado / "auditoria_base_modelado_v2.csv"
ruta_auditoria_vectores = carpeta_metadata_modelado / "auditoria_vectores_modelado_v2.csv"
ruta_auditoria_componentes_locales = carpeta_metadata_modelado / "auditoria_componentes_locales_v2.csv"
ruta_resumen_componentes_locales = carpeta_metadata_modelado / "resumen_componentes_locales_v2.csv"
ruta_resumen_grupos = carpeta_metadata_modelado / "resumen_grupos_brillo_ml_v2.csv"

df_base_modelado_final.to_parquet(ruta_modelado_completo_parquet, index=False)
df_modelado_entrenamiento.to_parquet(ruta_entrenamiento_parquet, index=False)
df_modelado_prueba.to_parquet(ruta_prueba_parquet, index=False)
df_modelado_entrenamiento.to_csv(ruta_entrenamiento_csv, index=False, date_format="%Y-%m-%dT%H:%M:%S%z")
df_modelado_prueba.to_csv(ruta_prueba_csv, index=False, date_format="%Y-%m-%dT%H:%M:%S%z")
df_diccionario_modelado.to_csv(ruta_diccionario_modelado, index=False)
df_configuracion_modelado.to_csv(ruta_configuracion_modelado, index=False)
df_parametros_centro_train.to_csv(ruta_parametros_centro, index=False)
df_resumen_particiones_modelado.to_csv(ruta_resumen_particiones, index=False)
df_auditoria_base_modelado.to_csv(ruta_auditoria_base, index=False)
df_auditoria_vectores.to_csv(ruta_auditoria_vectores, index=False)
df_auditoria_componentes_locales.to_csv(ruta_auditoria_componentes_locales, index=False)
df_resumen_componentes_locales.to_csv(ruta_resumen_componentes_locales, index=False)
df_resumen_grupos_ml.to_csv(ruta_resumen_grupos, index=False)

rutas_generadas_modelado = [
    ruta_modelado_completo_parquet,
    ruta_entrenamiento_parquet,
    ruta_entrenamiento_csv,
    ruta_prueba_parquet,
    ruta_prueba_csv,
    ruta_diccionario_modelado,
    ruta_configuracion_modelado,
    ruta_parametros_centro,
    ruta_resumen_particiones,
    ruta_auditoria_base,
    ruta_auditoria_vectores,
    ruta_auditoria_componentes_locales,
    ruta_resumen_componentes_locales,
    ruta_resumen_grupos
]

print("Archivos guardados:")

for ruta in rutas_generadas_modelado:
    print(ruta.relative_to(raiz_proyecto))

Archivos guardados:
data/processed/geminidas_modelado_v2/geminidas_modelado_completo_v2.parquet
data/processed/geminidas_modelado_v2/geminidas_modelado_entrenamiento_2019_2024_v2.parquet
data/processed/geminidas_modelado_v2/geminidas_modelado_entrenamiento_2019_2024_v2.csv
data/processed/geminidas_modelado_v2/geminidas_modelado_prueba_2025_v2.parquet
data/processed/geminidas_modelado_v2/geminidas_modelado_prueba_2025_v2.csv
data/processed/geminidas_modelado_v2/metadata/diccionario_variables_modelado_v2.csv
data/processed/geminidas_modelado_v2/metadata/configuracion_modelado_v2.csv
data/processed/geminidas_modelado_v2/metadata/parametros_centro_entrenamiento_v2.csv
data/processed/geminidas_modelado_v2/metadata/resumen_particiones_modelado_v2.csv
data/processed/geminidas_modelado_v2/metadata/auditoria_base_modelado_v2.csv
data/processed/geminidas_modelado_v2/metadata/auditoria_vectores_modelado_v2.csv
data/processed/geminidas_modelado_v2/metadata/auditoria_componentes_locales_v2.csv
data

In [33]:
# Recargar y verificar los archivos guardados y generar el manifiesto de integridad SHA-256

import hashlib

def normalizar_texto(serie):
    return serie.astype("string").fillna("<NA>").to_numpy(dtype=str)

def verificar_base_guardada(nombre, df_leido, df_original):
    error_componentes_locales = np.max(
        np.abs(
            df_leido[columnas_pca].to_numpy(dtype=float)
            - df_original[columnas_pca].to_numpy(dtype=float)
        )
    )

    error_componentes_cartesianas = np.max(
        np.abs(
            df_leido[["delta_v_x_km_s", "delta_v_y_km_s", "delta_v_z_km_s"]].to_numpy(dtype=float)
            - df_original[["delta_v_x_km_s", "delta_v_y_km_s", "delta_v_z_km_s"]].to_numpy(dtype=float)
        )
    )

    error_norma = np.max(
        np.abs(
            df_leido["norma_delta_v_km_s"].to_numpy(dtype=float)
            - df_original["norma_delta_v_km_s"].to_numpy(dtype=float)
        )
    )

    objetivos_leidos = pd.to_numeric(df_leido["objetivo_ml"], errors="coerce").to_numpy(dtype=float)
    objetivos_originales = pd.to_numeric(df_original["objetivo_ml"], errors="coerce").to_numpy(dtype=float)

    return {
        "archivo": nombre,
        "filas_correctas": len(df_leido) == len(df_original),
        "columnas_correctas": df_leido.columns.tolist() == df_original.columns.tolist(),
        "identificadores_correctos": np.array_equal(df_leido["id_fila_origen"].to_numpy(), df_original["id_fila_origen"].to_numpy()),
        "particiones_correctas": np.array_equal(normalizar_texto(df_leido["particion_modelado"]), normalizar_texto(df_original["particion_modelado"])),
        "grupos_brillo_correctos": np.array_equal(normalizar_texto(df_leido["grupo_brillo_ml"]), normalizar_texto(df_original["grupo_brillo_ml"])),
        "objetivos_correctos": np.allclose(objetivos_leidos, objetivos_originales, equal_nan=True),
        "error_maximo_componentes_locales": error_componentes_locales,
        "error_maximo_componentes_cartesianas": error_componentes_cartesianas,
        "error_maximo_norma": error_norma
    }

df_completo_recargado = pd.read_parquet(ruta_modelado_completo_parquet)
df_entrenamiento_parquet_recargado = pd.read_parquet(ruta_entrenamiento_parquet)
df_prueba_parquet_recargado = pd.read_parquet(ruta_prueba_parquet)
df_entrenamiento_csv_recargado = pd.read_csv(ruta_entrenamiento_csv)
df_prueba_csv_recargado = pd.read_csv(ruta_prueba_csv)
df_diccionario_recargado = pd.read_csv(ruta_diccionario_modelado)
df_configuracion_recargada = pd.read_csv(ruta_configuracion_modelado)

df_verificacion_guardado = pd.DataFrame([
    verificar_base_guardada("Completa Parquet", df_completo_recargado, df_base_modelado_final),
    verificar_base_guardada("Entrenamiento Parquet", df_entrenamiento_parquet_recargado, df_modelado_entrenamiento),
    verificar_base_guardada("Prueba Parquet", df_prueba_parquet_recargado, df_modelado_prueba),
    verificar_base_guardada("Entrenamiento CSV", df_entrenamiento_csv_recargado, df_modelado_entrenamiento),
    verificar_base_guardada("Prueba CSV", df_prueba_csv_recargado, df_modelado_prueba)
])

diccionario_correcto = df_diccionario_recargado["variable"].tolist() == columnas_base_modelado
version_configuracion = df_configuracion_recargada.loc[df_configuracion_recargada["elemento"] == "Versión", "valor"].iloc[0]
configuracion_correcta = version_configuracion == "geminidas_modelado_v2"

columnas_booleanas_verificacion = [
    "filas_correctas",
    "columnas_correctas",
    "identificadores_correctos",
    "particiones_correctas",
    "grupos_brillo_correctos",
    "objetivos_correctos"
]

if not df_verificacion_guardado[columnas_booleanas_verificacion].to_numpy().all():
    raise ValueError("Alguna base guardada no coincide con su versión en memoria.")

if df_verificacion_guardado[
    ["error_maximo_componentes_locales", "error_maximo_componentes_cartesianas", "error_maximo_norma"]
].to_numpy(dtype=float).max() > 1e-12:
    raise ValueError("La escritura o lectura introdujo diferencias numéricas mayores que la tolerancia.")

if not diccionario_correcto:
    raise ValueError("El diccionario guardado no coincide con las columnas de la base.")

if not configuracion_correcta:
    raise ValueError("La configuración guardada no identifica correctamente la versión v2.")

ruta_verificacion_guardado = carpeta_metadata_modelado / "verificacion_guardado_modelado_v2.csv"
df_verificacion_guardado.to_csv(ruta_verificacion_guardado, index=False)

def calcular_sha256(ruta):
    sha256 = hashlib.sha256()

    with ruta.open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            sha256.update(bloque)

    return sha256.hexdigest()

rutas_para_manifiesto = [*rutas_generadas_modelado, ruta_verificacion_guardado]

df_manifiesto_integridad = pd.DataFrame([
    {
        "archivo": str(ruta.relative_to(carpeta_modelado)),
        "tamano_bytes": ruta.stat().st_size,
        "sha256": calcular_sha256(ruta)
    }
    for ruta in rutas_para_manifiesto
])

ruta_manifiesto_integridad = carpeta_metadata_modelado / "manifiesto_integridad_modelado_v2.csv"
df_manifiesto_integridad.to_csv(ruta_manifiesto_integridad, index=False)

display(df_verificacion_guardado)
print("Diccionario correcto:", diccionario_correcto)
print("Configuración correcta:", configuracion_correcta)
print("Archivos incluidos en el manifiesto:", len(df_manifiesto_integridad))
display(df_manifiesto_integridad)
print("Manifiesto guardado en:", ruta_manifiesto_integridad.relative_to(raiz_proyecto))

,archivo,filas_correctas,columnas_correctas,identificadores_correctos,particiones_correctas,grupos_brillo_correctos,objetivos_correctos,error_maximo_componentes_locales,error_maximo_componentes_cartesianas,error_maximo_norma
0,Completa Parquet,True,True,True,True,True,True,0.0,0.0,0.0
1,Entrenamiento Parquet,True,True,True,True,True,True,0.0,0.0,0.0
2,Prueba Parquet,True,True,True,True,True,True,0.0,0.0,0.0
3,Entrenamiento CSV,True,True,True,True,True,True,0.0,0.0,0.0
4,Prueba CSV,True,True,True,True,True,True,0.0,0.0,0.0


Diccionario correcto: True
Configuración correcta: True
Archivos incluidos en el manifiesto: 15


,archivo,tamano_bytes,sha256
0,geminidas_modelado_completo_v2.parquet,4109632,21fbcd34076db40938a67486281a4c02e646388b092598...
1,geminidas_modelado_entrenamiento_2019_2024_v2....,2767324,98b51bdbc06aff64bd9737deb64c19a637efeebae55463...
2,geminidas_modelado_entrenamiento_2019_2024_v2.csv,6523272,ca33715122a01a114b005f05bc97b8a320acc0398301d9...
3,geminidas_modelado_prueba_2025_v2.parquet,1362485,a1feec51503dc8cc01a374026928e5452b1ec98a1ea528...
4,geminidas_modelado_prueba_2025_v2.csv,3112401,619bdf959390af0eaef4c84d9c7fd511e19b8140b38121...
5,metadata/diccionario_variables_modelado_v2.csv,5971,7a0fc145833fa7d73171625e245c99d2c8606cbe10c019...
6,metadata/configuracion_modelado_v2.csv,2490,c761bcc8455fdc7408c630d3146a85e6db6b5c76ff193d...
7,metadata/parametros_centro_entrenamiento_v2.csv,446,444d07ff95b6421707a5440b85b5fe061345e794b2fce7...
8,metadata/resumen_particiones_modelado_v2.csv,223,69e6f3d8f74f27bc32fa84aa725e8ab8eeac3271eb3c2b...
9,metadata/auditoria_base_modelado_v2.csv,245,259320ab8021e2c90015cc74b626fb1371a3485d4c3cd1...


Manifiesto guardado en: data/processed/geminidas_modelado_v2/metadata/manifiesto_integridad_modelado_v2.csv
